In [ ]:
# Cell 1: Install dependencies
print("Installing dependencies...")
%pip install chromadb pydantic PyPDF2 httpx anthropic

In [ ]:
# Cell 2: Load WHO MEC knowledge base into ChromaDB
import sys
import os
sys.path.append('..')

from ingest import ingest_knowledge_base
print("Loading WHO MEC knowledge base...")
success = ingest_knowledge_base()
print(f"Ingestion successful: {success}")

In [ ]:
# Cell 3: Define the safety screen function with a test profile
import sys
sys.path.append('..')

from engine.safety_screen import safety_screen

# Define a test profile matching one of the sample profiles
test_profile = {
  "age_group": "25-34",
  "ever_pregnant": True,
  "breastfeeding": True,
  "breastfeeding_infant_months": 3,
  "health_flags": [],
  "pregnancy_goal": "spacing_1_3_years",
  "partner_situation": "regular_partner",
  "prior_contraception": "none",
  "access": "clinic_accessible",
  "discretion_needed": False
}

print("Test profile defined:")
import json
print(json.dumps(test_profile, indent=2))

In [ ]:
# Cell 4: Run the scoring engine on that profile
from engine.scorer import score_methods

# Run the safety screen
eliminated = safety_screen(test_profile)
print(f"Eliminated methods: {[e['method'] for e in eliminated]}\n")

# Run scoring engine
scores = score_methods(test_profile, eliminated)
print(f"Top recommended methods: {scores[:3]}")

In [ ]:
# Cell 5: Call Claude API with the injected prompt
import os
from anthropic import Anthropic

# System and User prompt construction
system_prompt = "You are ContraBot, a contraception counselor."
user_prompt = f"""User profile:
- Name: unknown
- Gender: female
- Age: 30
- Breastfeeding (<6mo): True
- Health risk (HTN/migraine/clots): False
- Preference: spacing_1_3_years
- Clinic access: True
- Output Language requested: english

Eliminated methods (MEC): COC, combined_patch
Ranked methods:
1. Progestogen-only pill (score: 0.90)
2. Copper IUD (score: 0.86)
3. Injectable (DMPA) (score: 0.82)

Write a friendly recommendation in english in under 500 characters."""

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("Warning: ANTHROPIC_API_KEY not found in environment. Using mock recommendation.")
    recommendation_text = "Based on your profile, POP (daily pill), Copper IUD, and DMPA injection are highly suitable. Avoid combined pill/patch due to breastfeeding under 6 months. Please consult a healthcare worker."
else:
    try:
        client = Anthropic(api_key=api_key)
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=500,
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_prompt}
            ]
        )
        recommendation_text = response.content[0].text
    except Exception as e:
        print(f"Error calling Anthropic API: {e}")
        recommendation_text = "Fallback: Progestogen-only pill, Copper IUD, or DMPA are recommended for spacing 1-3 years while breastfeeding. Visit a clinic to confirm."

In [ ]:
# Cell 6: Show the output
print("--- ContraBot Recommendation Output ---")
print(recommendation_text)